# 📘 Semaine 9 — ADC + DMA

**Cours :** Microcontrôleurs STM32F103C6T6  
**Durée :** 4h30 (1h30 cours + 1h30 atelier + 1h30 homework)  
**Enseignant :** ____________________  
**Étudiant :** ____________________  
**Date :** ____________________

---

## 🎯 Objectifs pédagogiques de la semaine

À la fin de cette semaine, l'étudiant sera capable de :

1. **Expliquer** le principe du DMA et son intérêt pour l'ADC.
2. **Configurer** un transfert DMA ADC → mémoire en mode circulaire.
3. **Déclencher** une conversion ADC par un timer (acquisition périodique).
4. **Implémenter** un filtrage numérique (moyenne glissante, médiane).
5. **Détecter** un seuil sur un signal analogique en temps réel.

---

## 🗺️ Plan de la semaine

| Partie | Contenu | Durée |
|---|---|---|
| **A — Cours** | Activités 1 à 6 | 1h30 |
| **B — Atelier** | TP9 : acquisition périodique + seuil | 1h30 |
| **C — Homework** | Exercices 1 à 3 | 1h30 |
| **D — Auto-évaluation** | Checklist finale | 5 min |

---
# 🎓 PARTIE A — COURS INTÉGRÉ (1h30)

## 🔹 Activité 1 — Rappel & mise en contexte (10 min)

### 🔄 Rappel de la semaine 8
- **ADC** : 12 bits, 10 canaux, V_ref = VDDA.
- **Formules** : `Code = V_in/V_ref × 2^n`, `V_in = Code × V_ref / 2^n`.
- **Modes** : simple, continu, scan.
- **HAL** : `HAL_ADC_Start()`, `PollForConversion()`, `GetValue()`.

### ✍️ Questions flash (2 min)
1. Quelle est la résolution de l'ADC du STM32F103 ? → ...
2. Pour V_ref = 3.3 V, quelle est la valeur du LSB ? → ...
3. Que fait `HAL_ADC_PollForConversion()` ? → ...
4. Combien de canaux ADC externes ? → ...

### 🎯 Nouveau problème à résoudre
Comment acquérir **en continu** un signal à 1 kHz **sans occuper le CPU** ?

---

## 🔹 Activité 2 — Le DMA : principes (20 min)

### 📖 2.1 — Définition

Le **DMA** (*Direct Memory Access*) est un contrôleur qui effectue des **transferts de données** entre la mémoire et les périphériques **sans intervention du CPU**.

```
Sans DMA :                Avec DMA :
                                 
  ADC ──▶ CPU ──▶ RAM       ADC ──▶ DMA ──▶ RAM
         (occupé)                  (CPU libre)
```

### 📖 2.2 — Avantages

| Avantage | Description |
|---|---|
| **CPU libre** | Le CPU peut faire autre chose |
| **Vitesse** | Transferts rapides (1 transfert / cycle) |
| **Précision** | Aucun retard logiciel |
| **Énergie** | Moins de réveils CPU |
| **Simplicité** | Aucune ISR par échantillon |

### 📖 2.3 — DMA sur le STM32F103C6T6

| Paramètre | Valeur |
|---|---|
| Contrôleur | **DMA1** |
| Nombre de canaux | **7** |
| Sources | ADC1, SPI1/2, I2C1, USART1/2/3, TIM1/2/3/4 |
| Destinations | SRAM, Flash (non), périphériques |
| Modes | Normal, circulaire |
| Taille transfert | 8, 16 ou 32 bits |
| Incrémentation | Source et/ou destination |

**Mapping ADC1 → DMA :**

| Périphérique | Canal DMA |
|---|---|
| ADC1 | **Canal 1** |
| SPI1_RX | Canal 2 |
| SPI1_TX | Canal 3 |
| I2C1_RX | Canal 7 |
| USART1_TX | Canal 4 |
| USART1_RX | Canal 5 |

### 🐍 Simulation Python — Comparaison CPU : Polling vs DMA (10 min)

In [ ]:
# ============================================================
# Comparaison polling vs DMA : occupation CPU
# ============================================================

def occupation_polling(f_echantillonnage_hz, cycles_par_echantillon=200, f_cpu_hz=72e6):
    """
    Estime le % CPU occupé par une acquisition en polling.
    - cycles_par_echantillon : temps de lecture + traitement + boucle
    """
    cycles_par_seconde = f_echantillonnage_hz * cycles_par_echantillon
    return cycles_par_seconde / f_cpu_hz * 100

def occupation_dma(f_echantillonnage_hz, cycles_par_bloc=500, taille_bloc=100, f_cpu_hz=72e6):
    """
    Estime le % CPU avec DMA.
    Le CPU ne traite que les blocs (une interruption par bloc).
    """
    nb_blocs_s = f_echantillonnage_hz / taille_bloc
    cycles_par_seconde = nb_blocs_s * cycles_par_bloc
    return cycles_par_seconde / f_cpu_hz * 100

F_CPU = 72e6

print("⚙️  Occupation CPU selon la fréquence d'échantillonnage\n")
print(f"{'F_ech (Hz)':<14}{'Polling (%)':<16}{'DMA (%)':<14}{'Gain'}")
print("-" * 60)
for f in [10, 100, 1000, 5000, 10000, 50000, 100000]:
    p = occupation_polling(f)
    d = occupation_dma(f)
    gain = p / d if d > 0 else float('inf')
    print(f"{f:<14}{p:<16.3f}{d:<14.4f}x{gain:.0f}")

print("\n📌 À retenir :")
print("  → En dessous de 1 kHz, le polling est acceptable.")
print("  → Au-delà, le DMA devient indispensable.")
print("  → Le DMA libère le CPU pour d'autres tâches (calculs, communication).")

---

## 🔹 Activité 3 — Configuration ADC + DMA (20 min)

### 📖 3.1 — Chaîne complète

```
   TIM2 (TRGO) ──▶ ADC1 ──▶ DMA1 (canal 1) ──▶ SRAM (buffer)
                       │                              │
                       │ EOC                          │ IRQ (HT/TC)
                       ▼                              ▼
                    (interne)                      CPU traite
```

### 📖 3.2 — Configuration CubeMX

**1. ADC1 :**
- Mode : `Independent mode`
- Clock Prescaler : `/6` → 12 MHz
- Resolution : `12 bits`
- Scan Conversion Mode : `Disabled` (1 seul canal)
- Continuous Conversion Mode : `Disabled`
- DMA Continuous Requests : `Enabled`
- External Trigger Conversion Source : `Timer 2 Trigger Out event`
- Sampling Time : `7.5 Cycles`

**2. DMA Settings :**
- Cliquer sur *Add* → ADC1
- Mode : `Circular`
- Data Width : `Half Word` (16 bits)
- Increment Memory : `Enabled`
- Increment Peripheral : `Disabled`
- Priority : `High`

**3. TIM2 :**
- Mode : base de temps (S5)
- Trigger Event Selection : `Update Event`
- PSC = 71, ARR = 999 → 1 kHz

### 📖 3.3 — Code HAL

In [ ]:
/* ============================================================
   ADC1 + DMA circulaire + trigger TIM2
   ============================================================ */

#include "main.h"

#define TAILLE_BUFFER  100

ADC_HandleTypeDef hadc1;
DMA_HandleTypeDef hdma_adc1;
TIM_HandleTypeDef htim2;

volatile uint16_t adc_buffer[TAILLE_BUFFER];
volatile uint8_t  demi_buffer_pret = 0;
volatile uint8_t  buffer_complet   = 0;

int main(void)
{
    HAL_Init();
    SystemClock_Config();
    MX_GPIO_Init();
    MX_DMA_Init();
    MX_ADC1_Init();
    MX_TIM2_Init();
    MX_USART2_UART_Init();

    HAL_ADCEx_Calibration_Start(&hadc1);

    // Démarrer TIM2 en TRGO
    HAL_TIM_Base_Start(&htim2);

    // Démarrer ADC + DMA circulaire
    HAL_ADC_Start_DMA(&hadc1,
                      (uint32_t*)adc_buffer,
                      TAILLE_BUFFER);

    while (1)
    {
        // Traitement par bloc (drapeau levé par ISR DMA)
        if (demi_buffer_pret || buffer_complet)
        {
            // Calcul moyenne sur 100 échantillons
            uint32_t somme = 0;
            for (uint16_t i = 0; i < TAILLE_BUFFER; i++)
                somme += adc_buffer[i];
            uint16_t moyenne = somme / TAILLE_BUFFER;

            // Conversion en tension
            float tension = moyenne * 3.3f / 4095.0f;

            char buf[64];
            int n = snprintf(buf, sizeof(buf),
                             "Moyenne ADC=%4u  V=%.4f V\r\n",
                             moyenne, tension);
            HAL_UART_Transmit(&huart2, (uint8_t*)buf, n, 100);

            demi_buffer_pret = 0;
            buffer_complet   = 0;
        }
    }
}

/* --- Callbacks DMA --- */
void HAL_ADC_ConvHalfCpltCallback(ADC_HandleTypeDef *hadc)
{
    demi_buffer_pret = 1;
}

void HAL_ADC_ConvCpltCallback(ADC_HandleTypeDef *hadc)
{
    buffer_complet = 1;
}

### 🔍 Analyse du code

| Élément | Rôle |
|---|---|
| `HAL_ADC_Start_DMA()` | Démarre ADC + DMA en une seule fonction |
| Buffer circulaire | Rempli en boucle par le DMA |
| `ConvHalfCpltCallback` | Appelé quand la moitié du buffer est remplie |
| `ConvCpltCallback` | Appelé quand le buffer est entièrement rempli |
| `volatile` | Variables modifiées par ISR DMA |
| Double buffering | Traiter un demi-buffer pendant que l'autre se remplit |

### 📖 3.4 — Version LL (rapide)

```c
LL_DMA_SetDataLength(DMA1, LL_DMA_CHANNEL_1, TAILLE_BUFFER);
LL_DMA_EnableChannel(DMA1, LL_DMA_CHANNEL_1);
LL_ADC_REG_SetDMATransfer(ADC1, LL_ADC_REG_DMA_TRANSFER_UNLIMITED);
LL_ADC_Enable(ADC1);
LL_ADC_REG_StartConversionSWStart(ADC1);
```

---

## 🔹 Activité 4 — Trigger par timer (15 min)

### 📖 4.1 — Pourquoi un timer ?

Un trigger logiciel (SWSTART) démarre une conversion **quand on le décide**, mais avec **jitter**.  
Un trigger matériel (timer) démarre une conversion **périodiquement et précisément**.

```
Trigger logiciel :        Trigger timer :
                                 
  t1  t2   t3   t4          t1  t2  t3  t4
  |   |    |    |            |   |   |   |
  (jitter)                  (précis)
```

### 📖 4.2 — Configuration TIM2 en TRGO

Dans CubeMX :
- TIM2 → Clock Source = Internal Clock
- PSC = 71, ARR = 999 → 1 kHz
- Trigger Event Selection : `Update Event`

**Chaîne interne :**
```
TIM2 --UIF--> TRGO --> ADC1 --EOC--> DMA --> SRAM
```

> 💡 **Résultat** : une conversion ADC exactement à chaque débordement du timer.

---

## 🔹 Activité 5 — Filtrage numérique (15 min)

### 📖 5.1 — Pourquoi filtrer ?

Un ADC réel produit toujours du **bruit** :
- Bruit thermique des composants
- Bruit de l'alimentation (VDDA)
- Interférences électromagnétiques
- Quantification (LSB)

### 📖 5.2 — Filtres simples

| Filtre | Complexité | Effet |
|---|---|---|
| **Moyenne glissante** | ★☆☆ | Réduit bruit blanc |
| **Médiane** | ★★☆ | Rejette pics isolés |
| **Passe-bas IIR** | ★★☆ | Lisse, temps réel |
| **Kalman** | ★★★ | Optimal, complexe |
| **Filtre FIR** | ★★☆ | Linéaire, contrôle fréquentiel |

**Moyenne glissante (N échantillons) :**
```
y[n] = (x[n] + x[n-1] + ... + x[n-N+1]) / N
```
Le bruit est réduit d'un facteur **√N**.

**Filtre IIR (passe-bas 1er ordre) :**
```
y[n] = α × x[n] + (1 - α) × y[n-1]
α ∈ [0, 1]  →  plus α est petit, plus le filtrage est fort
```

### 📖 5.3 — Détection de seuil (hystérésis)

Pour éviter le **rebond** autour d'un seuil :

```
  ON ◄─── si V > V_haut
  OFF ◄── si V < V_bas

  Zone morte entre V_bas et V_haut → état conservé
```

Exemple : seuil haut = 2.0 V, seuil bas = 1.8 V

### 🐍 Simulation Python — Filtres numériques (15 min)

In [ ]:
# ============================================================
# Filtres : moyenne glissante, médiane, IIR passe-bas
# ============================================================

import random, statistics
random.seed(1)

def signal_bruite(n, valeur_reelle=1.5, bruit_mv=50):
    """Génère un signal ADC bruité (en volts)."""
    return [valeur_reelle + random.gauss(0, bruit_mv / 1000) for _ in range(n)]

def moyenne_glissante(s, N):
    y = []
    for i in range(len(s)):
        debut = max(0, i - N + 1)
        y.append(sum(s[debut:i+1]) / len(s[debut:i+1]))
    return y

def mediane_glissante(s, N):
    y = []
    for i in range(len(s)):
        debut = max(0, i - N + 1)
        y.append(statistics.median(s[debut:i+1]))
    return y

def iir_passe_bas(s, alpha):
    y = [s[0]]
    for x in s[1:]:
        y.append(alpha * x + (1 - alpha) * y[-1])
    return y

# Signal test
V_REEL = 1.5
N      = 50
s = signal_bruite(N, V_REEL, bruit_mv=80)

moy  = moyenne_glissante(s, 10)
med  = mediane_glissante(s, 10)
iir1 = iir_passe_bas(s, 0.2)
iir2 = iir_passe_bas(s, 0.05)

print(f"🎯 Signal cible : {V_REEL} V  (bruit ±80 mV)\n")
print(f"{'#':<4}{'Brut':<10}{'Moy(10)':<12}{'Méd(10)':<12}{'IIR a=0.2':<12}{'IIR a=0.05'}")
print("-" * 70)
for i in range(0, N, 5):
    print(f"{i:<4}{s[i]:<10.4f}{moy[i]:<12.4f}{med[i]:<12.4f}{iir1[i]:<12.4f}{iir2[i]:.4f}")

print(f"\n📊 Écart-type par méthode (mV) :")
for nom, sig in [('Brut', s), ('Moy(10)', moy), ('Méd(10)', med),
                 ('IIR 0.2', iir1), ('IIR 0.05', iir2)]:
    print(f"  {nom:<12} : {statistics.stdev(sig)*1000:.3f} mV")

### 🐍 Simulation Python — Détection de seuil avec hystérésis (10 min)

In [ ]:
# ============================================================
# Détection de seuil avec et sans hystérésis
# ============================================================

import random
random.seed(3)

def detection_simple(signal, seuil):
    """Détection simple (peut osciller autour du seuil)."""
    return [1 if v > seuil else 0 for v in signal]

def detection_hysteresis(signal, seuil_haut, seuil_bas):
    """Détection avec hystérésis (état stable)."""
    etat = 0
    sortie = []
    for v in signal:
        if v > seuil_haut:
            etat = 1
        elif v < seuil_bas:
            etat = 0
        sortie.append(etat)
    return sortie

# Signal qui oscille autour de 1.9 V (bruit + transition lente)
signal = []
for i in range(60):
    base = 1.5 + i * 0.02   # montée progressive 1.5 → 2.7 V
    signal.append(base + random.gauss(0, 0.05))

SEUIL = 2.0
V_HAUT = 2.05
V_BAS  = 1.95

simple = detection_simple(signal, SEUIL)
hyst   = detection_hysteresis(signal, V_HAUT, V_BAS)

def compte_transitions(sig):
    n = 0
    for i in range(1, len(sig)):
        if sig[i] != sig[i-1]:
            n += 1
    return n

print(f"{'i':<4}{'V (V)':<10}{'Simple':<10}{'Hystérésis'}")
print("-" * 40)
for i in range(0, 60, 3):
    print(f"{i:<4}{signal[i]:<10.4f}{simple[i]:<10}{hyst[i]}")

print(f"\n📊 Transitions (rebonds) :")
print(f"  Simple    : {compte_transitions(simple)} transitions")
print(f"  Hystérésis: {compte_transitions(hyst)} transitions")
print(f"\n📌 L'hystérésis élimine les oscillations autour du seuil.")

---

## 🔹 Activité 6 — QCM formatif (10 min)

**1. Combien de canaux DMA possède le DMA1 du STM32F103 ?**  
A. 4  
B. 7  
C. 12  
D. 16

**2. Sur quel canal DMA est mappé ADC1 ?**  
A. Canal 1  
B. Canal 2  
C. Canal 3  
D. Canal 7

**3. Le mode DMA circulaire permet :**  
A. Un seul transfert  
B. Un remplissage en boucle du buffer  
C. Un transfert mémoire → mémoire uniquement  
D. Le transfert de 8 bits seulement

**4. Le callback appelé quand la moitié du buffer DMA est remplie est :**  
A. `HAL_ADC_ConvCpltCallback`  
B. `HAL_ADC_ConvHalfCpltCallback`  
C. `HAL_DMA_CompleteCallback`  
D. `HAL_DMA_ErrorCallback`

**5. Le gain en occupation CPU du DMA par rapport au polling est :**  
A. Nul  
B. Faible  
C. Important  
D. Négatif

**6. L'hystérésis sert à :**  
A. Accélérer l'ADC  
B. Éviter les oscillations autour d'un seuil  
C. Filtrer le bruit  
D. Augmenter la résolution

### ✅ Corrigé du QCM formatif

| Q | Réponse | Explication |
|---|---|---|
| 1 | **B — 7** | DMA1 a 7 canaux |
| 2 | **A — Canal 1** | ADC1 → DMA1_Channel1 |
| 3 | **B — Remplissage en boucle** | Mode circulaire |
| 4 | **B — `HAL_ADC_ConvHalfCpltCallback`** | Callback Half-Complete |
| 5 | **C — Important** | Gain ×10 à ×1000 |
| 6 | **B — Éviter oscillations** | Zone morte entre 2 seuils |

**Mon score : ___ / 6**

---

# 🛠️ PARTIE B — ATELIER / TP (1h30)

## 🧪 TP9 — Acquisition périodique + détection de seuil

### 🎯 Objectif
Acquérir un signal analogique à 1 kHz via ADC + DMA + TIM2, calculer la moyenne glissante, détecter un seuil et envoyer les données sur UART.

### 📋 Tâches à réaliser (par binôme)

| # | Tâche | Durée | Livrable |
|---|---|---|---|
| 1 | Configurer ADC1_IN0 + DMA circulaire (buffer 100) | 15 min | Capture CubeMX |
| 2 | Configurer TIM2 en TRGO à 1 kHz | 10 min | Capture |
| 3 | Configurer USART2 (115200) | 5 min | Capture |
| 4 | Implémenter moyenne glissante + envoi UART | 20 min | Code |
| 5 | Détection de seuil 2 V avec hystérésis | 15 min | Démo LED |
| 6 | Envoyer les données au PC pour tracé (SerialPlot) | 15 min | Capture graphique |
| 7 | Rédiger le compte-rendu | 10 min | CR |

### ⚙️ Code complet — Acquisition + filtrage + seuil

In [ ]:
/* ============================================================
   TP9 - ADC + DMA + TIM2 + seuil + UART
   - PA0 : ADC_IN0 (signal analogique)
   - TIM2 : trigger ADC à 1 kHz
   - DMA1 canal 1 : buffer circulaire de 100 uint16_t
   - USART2 : envoi des données
   - PC13 : LED état du seuil
   ============================================================ */

#include "main.h"

#define TAILLE_BUFFER   100
#define SEUIL_HAUT      2480   // 2.0 V → 2.0/3.3 × 4096
#define SEUIL_BAS       2230   // 1.8 V → 1.8/3.3 × 4096

ADC_HandleTypeDef hadc1;
DMA_HandleTypeDef hdma_adc1;
TIM_HandleTypeDef htim2;
UART_HandleTypeDef huart2;

volatile uint16_t buffer_adc[TAILLE_BUFFER];
volatile uint8_t  drapeau_traitement = 0;

/* --- Moyenne glissante sur N échantillons --- */
static uint16_t moyenne_buffer(volatile uint16_t *buf, uint16_t n)
{
    uint32_t somme = 0;
    for (uint16_t i = 0; i < n; i++)
        somme += buf[i];
    return somme / n;
}

/* --- Traitement principal --- */
static void traiter_donnees(void)
{
    static uint8_t etat_led = 0;

    uint16_t moy = moyenne_buffer(buffer_adc, TAILLE_BUFFER);
    float tension = moy * 3.3f / 4095.0f;

    /* Détection de seuil avec hystérésis */
    if (moy > SEUIL_HAUT) {
        etat_led = 1;
    } else if (moy < SEUIL_BAS) {
        etat_led = 0;
    }
    HAL_GPIO_WritePin(GPIOC, GPIO_PIN_13,
                      etat_led ? GPIO_PIN_SET : GPIO_PIN_RESET);

    /* Envoi format SerialPlot : "moy tension\n" */
    char buf[48];
    int n = snprintf(buf, sizeof(buf), "%u %.4f\n", moy, tension);
    HAL_UART_Transmit(&huart2, (uint8_t*)buf, n, 100);
}

int main(void)
{
    HAL_Init();
    SystemClock_Config();
    MX_GPIO_Init();
    MX_DMA_Init();
    MX_ADC1_Init();
    MX_TIM2_Init();
    MX_USART2_UART_Init();

    HAL_ADCEx_Calibration_Start(&hadc1);

    HAL_TIM_Base_Start(&htim2);   // trigger TRGO

    HAL_ADC_Start_DMA(&hadc1,
                      (uint32_t*)buffer_adc,
                      TAILLE_BUFFER);

    while (1)
    {
        if (drapeau_traitement)
        {
            drapeau_traitement = 0;
            traiter_donnees();
        }
    }
}

/* --- Callback DMA demi-buffer --- */
void HAL_ADC_ConvHalfCpltCallback(ADC_HandleTypeDef *hadc)
{
    drapeau_traitement = 1;
}

/* --- Callback DMA buffer complet --- */
void HAL_ADC_ConvCpltCallback(ADC_HandleTypeDef *hadc)
{
    drapeau_traitement = 1;
}

### 🔍 Analyse du code

| Élément | Rôle |
|---|---|
| `HAL_ADC_Start_DMA()` | Démarre ADC + DMA |
| Buffer circulaire | Rempli en continu |
| Callback Half/Complete | Drapeau pour traitement hors ISR |
| `moyenne_buffer()` | Moyenne sur 100 échantillons |
| Hystérésis | 2 seuils pour éviter les rebonds |
| `SerialPlot` | Format "valeur1 valeur2\n" pour tracé |

### 📖 Configuration CubeMX — Résumé

**ADC1 :**
- IN0 activé (PA0)
- Mode : Independent
- Clock Prescaler : /6
- Resolution : 12 bits
- Continuous Conversion Mode : Disabled
- DMA Continuous Requests : Enabled
- External Trigger Conversion Source : Timer 2 Trigger Out event

**DMA Settings :**
- Ajouter ADC1
- Mode : Circular
- Data Width : Half Word (16 bits)
- Increment Memory : Enabled

**TIM2 :**
- PSC = 71, ARR = 999 → 1 kHz
- Trigger Event Selection : Update Event

### 🐍 Simulation Python — Tracé ASCII des données ADC (15 min)

Simulons l'acquisition et affichons les données sous forme graphique ASCII.

In [ ]:
# ============================================================
# Tracé ASCII d'un signal ADC acquis via DMA
# ============================================================

import math, random
random.seed(7)

def signal_sinusoidal(n, f_signal=2, f_ech=50, amplitude_v=1.5, offset_v=1.65, bruit_mv=30):
    """Simule un signal sinusoïdal échantillonné et bruité."""
    valeurs = []
    for i in range(n):
        t = i / f_ech
        v = offset_v + amplitude_v * math.sin(2 * math.pi * f_signal * t)
        v += random.gauss(0, bruit_mv/1000)
        v = max(0, min(3.3, v))
        valeurs.append(v)
    return valeurs

def tracer_ascii(valeurs, hauteur=12, largeur=70, v_min=0, v_max=3.3):
    """Trace un signal en ASCII."""
    pas = max(1, len(valeurs) // largeur)
    echantillons = valeurs[::pas][:largeur]
    
    # Grille
    grille = [[' '] * len(echantillons) for _ in range(hauteur)]
    for col, v in enumerate(echantillons):
        # Position verticale (0 en haut)
        y = int((1 - (v - v_min) / (v_max - v_min)) * (hauteur - 1))
        y = max(0, min(hauteur - 1, y))
        grille[y][col] = '●'
    
    print(f"  {v_max:.1f} V ┤" + "".join(grille[0]))
    for r in range(1, hauteur - 1):
        v_inter = v_max - (r / (hauteur - 1)) * (v_max - v_min)
        print(f"  {v_inter:4.2f} V ┤" + "".join(grille[r]))
    print(f"  {v_min:.1f} V ┤" + "".join(grille[hauteur - 1]))
    print("           " + "─" * len(echantillons))

# Générer 200 échantillons
ech = signal_sinusoidal(200, f_signal=2, f_ech=50, amplitude_v=1.4, offset_v=1.65)

print("📈 Signal ADC simulé (sinusoïde 2 Hz échantillonnée à 50 Hz)\n")
tracer_ascii(ech)

print(f"\n📊 Statistiques :")
print(f"  Min     : {min(ech):.4f} V")
print(f"  Max     : {max(ech):.4f} V")
print(f"  Moyenne : {sum(ech)/len(ech):.4f} V")
print(f"  Crête-à-crête : {max(ech)-min(ech):.4f} V")

### 🐍 Simulation Python — Détection de seuil sur signal sinusoïdal (10 min)

In [ ]:
# Détection de seuil sur un signal bruité

def detecter_seuils(signal, seuil_haut, seuil_bas):
    """Retourne les transitions de l'état (0 ou 1)."""
    etat = 0
    transitions = []
    for i, v in enumerate(signal):
        if v > seuil_haut and etat == 0:
            etat = 1
            transitions.append((i, "↑", v))
        elif v < seuil_bas and etat == 1:
            etat = 0
            transitions.append((i, "↓", v))
    return transitions

V_HAUT = 2.5
V_BAS  = 0.8

trans = detecter_seuils(ech, V_HAUT, V_BAS)

print(f"🎯 Détection de seuil (haut={V_HAUT} V, bas={V_BAS} V)\n")
print(f"Nombre de transitions : {len(trans)}\n")
print(f"{'#':<4}{'Index':<10}{'Sens':<8}{'V (V)'}")
print("-" * 35)
for i, (idx, sens, v) in enumerate(trans, 1):
    print(f"{i:<4}{idx:<10}{sens:<8}{v:.4f}")

### 📝 Compte-rendu de TP9

**Nom :** __________________  **Prénom :** __________________  **Binôme :** __________________

**1. Configuration CubeMX**
- ADC1 : canal = ... , trigger = ... , DMA = ...
- DMA : mode = ... , largeur = ... , incrément = ...
- TIM2 : PSC = ... , ARR = ... → F_trigger = ... Hz
- Taille buffer : ... échantillons
- USART2 : baudrate = ...

**2. Code ajouté dans `main.c`**
```c
// Colle ici ton code
```

**3. Observation UART (SerialPlot)**
- Fréquence d'envoi effective : ... Hz
- Signal visualisé : ...
- Bruit observé : ... mV

**4. Détection de seuil**
- Seuil haut utilisé : ... V
- Seuil bas utilisé : ... V
- LED s'allume correctement ? ...
- Y a-t-il des rebonds ? ...

**5. Comparaison avec/sans DMA**
- CPU occupé avec polling : ... %
- CPU occupé avec DMA : ... %
- Conclusion : ...

**6. Problèmes rencontrés**
- ...

**7. Solutions apportées**
- ...

### 🧪 Exercice bonus — Mini-projet d'acquisition

Réaliser un **oscilloscope simplifié** :
- ADC + DMA + TIM2 à 10 kHz
- Buffer de 500 échantillons
- Envoi binaire des données via UART (ou texte)
- Script Python côté PC pour tracer le signal reçu

**Bonus :** implémenter une détection de **fréquence** par comptage de passages par zéro.

In [ ]:
// Squelette solution bonus (côté STM32)

#define OSC_N  500
volatile uint16_t osc_buffer[OSC_N];
volatile uint8_t  osc_pret = 0;

void HAL_ADC_ConvCpltCallback(ADC_HandleTypeDef *hadc)
{
    osc_pret = 1;
}

// Dans le main :
// while (1) {
//   if (osc_pret) {
//     osc_pret = 0;
//     // Envoyer osc_buffer sur UART en binaire
//     HAL_UART_Transmit(&huart2, (uint8_t*)osc_buffer, OSC_N * 2, 1000);
//   }
// }

In [ ]:
# Squelette côté PC (Python) — réception UART + tracé

def script_pc_reception():
    """
    Ce script s'exécute sur le PC pour :
    - ouvrir le port série
    - lire 500 × 2 octets
    - tracer le signal
    """
    # import serial
    # import matplotlib.pyplot as plt
    #
    # ser = serial.Serial('/dev/ttyUSB0', 115200)
    # data = ser.read(1000)   # 500 uint16_t = 1000 octets
    # valeurs = [data[i] | (data[i+1] << 8) for i in range(0, 1000, 2)]
    # plt.plot(valeurs)
    # plt.show()
    pass

print("💡 Sur le PC, utiliser pyserial + matplotlib")

---

# 🏠 PARTIE C — HOMEWORK (1h30)

## 📚 Exercices à rendre

### 🧩 Exercice 1 — Configuration DMA (30 min)

Compléter le tableau suivant en indiquant la configuration DMA pour chaque scénario.

| # | Scénario | Canal DMA | Mode | Largeur | Incrément mémoire |
|---|---|---|---|---|---|
| 1 | ADC1 1 canal, buffer 100 | ? | ? | ? | ? |
| 2 | SPI1_RX, buffer 64 octets | ? | ? | ? | ? |
| 3 | USART1_TX, chaîne 32 octets | ? | ? | ? | ? |
| 4 | I2C1_RX, buffer 16 octets | ? | ? | ? | ? |
| 5 | ADC1 scan 4 canaux, buffer 4×10 | ? | ? | ? | ? |

### 🧩 Exercice 2 — Filtrage numérique (30 min)

Un ADC fournit les valeurs suivantes (12 bits, V_ref = 3.3 V) :

```
2048, 2050, 2045, 2100, 2048, 2047, 2052, 2049, 2048, 2048
```

**Questions :**
1. Calculer la moyenne. Quelle tension cela représente-t-il ?
2. Calculer la médiane.
3. Calculer l'écart-type.
4. Quelle valeur aberrante est présente ?
5. Quelle méthode élimine-t-elle le mieux cette valeur ?
6. Appliquer un filtre IIR avec α = 0.3 sur les 3 premières valeurs.
7. Quelle méthode recommanderais-tu pour un signal lent ?

In [ ]:
# Corrigé Exercice 2
import statistics

lectures = [2048, 2050, 2045, 2100, 2048, 2047, 2052, 2049, 2048, 2048]
V_REF = 3.3

moy     = statistics.mean(lectures)
med     = statistics.median(lectures)
sigma   = statistics.stdev(lectures)

print(f"1. Moyenne  : {moy:.2f}  ({moy/4096*V_REF:.4f} V)")
print(f"2. Médiane  : {med}  ({med/4096*V_REF:.4f} V)")
print(f"3. Écart-type : {sigma:.3f}")

# Détection d'outlier (|x - mean| > 2σ)
outliers = [i for i, v in enumerate(lectures) if abs(v - moy) > 2 * sigma]
print(f"4. Valeurs aberrantes (index) : {outliers}")
print(f"   → valeur = {lectures[outliers[0]]}" if outliers else "   → aucune")

print(f"5. La médiane élimine le mieux les pics isolés.")

# Filtre IIR
alpha = 0.3
y = [lectures[0]]
for x in lectures[1:4]:
    y.append(alpha * x + (1 - alpha) * y[-1])
print(f"6. IIR α=0.3 (3 premières valeurs) : {[f'{v:.2f}' for v in y]}")
print(f"7. Pour un signal lent : moyenne glissante ou filtre IIR (temps réel).")

### 🧩 Exercice 3 — Lecture du RM0008 (30 min)

Lire la section **DMA** du chapitre 10 (DMA controller) du RM0008 et répondre :

1. Combien de canaux DMA possède le DMA1 ? Sont-ils prioritaires entre eux ?
2. Que signifie **Circular Mode** et comment l'activer ?
3. Quels sont les 3 niveaux de priorité DMA ?
4. Que fait le bit **TCIF** dans le registre **ISR** ?
5. Quelle est la différence entre **half-transfer** et **transfer-complete** ?
6. Comment activer l'interruption DMA sur un canal ?

### ✍️ Réponses — Exercice 3

1. ...
2. ...
3. ...
4. ...
5. ...
6. ...

---

## 🧮 Exercice supplémentaire — Buffer double (optionnel)

Simuler en Python un système **double buffering** : pendant que le DMA remplit un buffer, le CPU traite l'autre.

**Implémentation :**
- Deux buffers de 100 échantillons
- Deux drapeaux : `buffer_0_pret`, `buffer_1_pret`
- Alterner entre les deux
- Mesurer la latence entre acquisition et traitement

In [ ]:
# Simulation double buffering en Python

import random
random.seed(5)

class DoubleBuffer:
    def __init__(self, taille=100):
        self.buffers = [[0]*taille, [0]*taille]
        self.prets   = [False, False]
        self.actif   = 0
        self.taille  = taille
        self.index   = 0

    def ecrire(self, val):
        """Simule l'écriture DMA dans le buffer courant."""
        self.buffers[self.actif][self.index] = val
        self.index += 1
        if self.index >= self.taille:
            self.prets[self.actif] = True
            self.actif = 1 - self.actif
            self.index = 0

    def traiter(self):
        """Retourne la moyenne du buffer prêt (à traiter)."""
        autre = 1 - self.actif
        if not self.prets[autre]:
            return None
        self.prets[autre] = False
        buf = self.buffers[autre]
        return sum(buf) / len(buf)

# Simulation : acquisition à 1 kHz
db = DoubleBuffer(100)
moyennes = []

for i in range(1000):
    val = 2000 + random.gauss(0, 50)
    db.ecrire(val)
    m = db.traiter()
    if m is not None:
        moyennes.append(m)

print(f"📊 Simulation double buffering sur 1000 acquisitions")
print(f"  Buffer taille : {db.taille}")
print(f"  Moyennes calculées : {len(moyennes)}")
print(f"  Moyenne globale : {sum(moyennes)/len(moyennes):.2f}")
print(f"  Écart-type : {statistics.stdev(moyennes):.3f}")

---
# ✅ PARTIE D — AUTO-ÉVALUATION Semaine 9

Coche ce que tu maîtrises.

- [ ] Je comprends le principe du DMA et ses avantages.
- [ ] Je connais le mapping ADC1 → DMA1_Channel1.
- [ ] Je connais les modes DMA (normal, circulaire).
- [ ] Je sais configurer ADC + DMA en CubeMX.
- [ ] Je sais utiliser `HAL_ADC_Start_DMA()`.
- [ ] Je connais les callbacks DMA (Half/Complete).
- [ ] Je sais configurer TIM2 en TRGO pour déclencher l'ADC.
- [ ] Je sais implémenter une moyenne glissante.
- [ ] Je sais implémenter un filtre IIR passe-bas.
- [ ] Je sais implémenter une détection de seuil avec hystérésis.
- [ ] J'ai envoyé les données ADC au PC pour tracé.
- [ ] J'ai rédigé mon compte-rendu de TP9.
- [ ] J'ai complété le tableau de configuration DMA.
- [ ] J'ai lu la section DMA du RM0008.

### 📊 Mon score : ___ / 14

| Score | Interprétation |
|---|---|
| 12–14 | ✅ Prêt pour la S10 (I2C/SPI) |
| 8–11 | ⚠️ Revoir les points manquants |
| < 8 | 🔁 Reprendre les activités 2 à 6 |

---
# 📚 RESSOURCES Semaine 9

### Documents officiels
- 📄 **RM0008** — chapitre 10 (DMA controller), chapitre 11 (ADC)
- 📄 **AN2834** — How to get the best ADC accuracy
- 📄 **AN3116** — ADC modes and their applications
- 📄 **UM1850** — HAL DMA documentation

### Outils
- **STM32CubeMX** — Analog → ADC1, DMA Settings, Timers
- **SerialPlot** ou **Python/matplotlib** pour visualiser
- **Oscilloscope** pour comparer signal réel vs données reçues

### Vidéos
- *STM32 ADC + DMA Tutorial* — ControllersTech
- *DMA Explained* — YouTube (ARM Education)

### Bonnes pratiques
- Toujours calibrer l'ADC avant la première conversion.
- Utiliser un trigger matériel (timer) pour l'acquisition périodique.
- Placer les buffers en `volatile` et `uint16_t` (12 bits → 16 bits).
- Faire un double buffering pour ne pas bloquer le DMA.
- Filtrer les valeurs (moyenne, médiane, IIR).
- Utiliser l'hystérésis pour les détections de seuil.
- Ne pas traiter les données dans l'ISR : poser un flag.

---

### 🔗 Passage à la semaine 10

**Prochaine séance :** Communication — exposés I2C et SPI  
- Bus I2C : 2 fils (SDA, SCL), adressage 7 bits, ACK/NACK
- Bus SPI : 4 fils (MOSI, MISO, SCK, CS), full-duplex
- Comparaison I2C / SPI
- TP : scan I2C + loopback SPI

**Préparation :**
- Préparer l'exposé I2C ou SPI (par groupe).
- Lire les chapitres 26 (I2C) et 25 (SPI) du RM0008.

---

**Fin du notebook — Semaine 9** ✨